# Risk Segmentation and Loan Decision Strategy

## Objective

This notebook uses Probability of Default (PD) scores generated from the LightGBM model to create borrower risk segments.

The goal is to convert model predictions into business-friendly credit risk outputs such as:

- Risk bands
- Risk scores
- Default rate summaries
- Preliminary loan decisions
- Decision-level risk summaries

This step connects machine learning model output to practical lending and portfolio risk decisions.

In [1]:
import pandas as pd
import numpy as np

## Import Scored Model Output

The scored output from the Probability of Default model is imported.

This dataset contains:

- original borrower and loan features
- actual default flag
- predicted PD score
- predicted default class

The PD score represents the model-estimated probability that a borrower may default.

In [4]:
pd.set_option('display.max_rows', 150)
pd.set_option('display.max_columns', 100)

## Importing scored outputs

In [3]:
df = pd.read_pickle('../data/pd_model_scored_output.pkl')
df.shape

(40000, 121)

## Review Available Columns

The column list is reviewed to understand the available borrower, loan, engineered, and model output variables.

This helps confirm that the dataset contains the required fields for risk segmentation.

In [6]:
df.columns.to_list()

['loan_amnt',
 'funded_amnt',
 'funded_amnt_inv',
 'term',
 'int_rate',
 'installment',
 'grade',
 'sub_grade',
 'emp_title',
 'emp_length',
 'home_ownership',
 'annual_inc',
 'verification_status',
 'issue_d',
 'purpose',
 'addr_state',
 'dti',
 'delinq_2yrs',
 'earliest_cr_line',
 'inq_last_6mths',
 'open_acc',
 'pub_rec',
 'revol_bal',
 'revol_util',
 'total_acc',
 'initial_list_status',
 'total_rec_late_fee',
 'last_pymnt_d',
 'last_credit_pull_d',
 'collections_12_mths_ex_med',
 'application_type',
 'acc_now_delinq',
 'tot_coll_amt',
 'tot_cur_bal',
 'total_rev_hi_lim',
 'acc_open_past_24mths',
 'avg_cur_bal',
 'bc_open_to_buy',
 'bc_util',
 'chargeoff_within_12_mths',
 'delinq_amnt',
 'mo_sin_old_il_acct',
 'mo_sin_old_rev_tl_op',
 'mo_sin_rcnt_rev_tl_op',
 'mo_sin_rcnt_tl',
 'mort_acc',
 'mths_since_recent_bc',
 'mths_since_recent_inq',
 'num_accts_ever_120_pd',
 'num_actv_bc_tl',
 'num_actv_rev_tl',
 'num_bc_sats',
 'num_bc_tl',
 'num_il_tl',
 'num_op_rev_tl',
 'num_rev_accts',

## Review Model Prediction Outputs

The key model output columns are reviewed:

- `actual_default_flag` → actual loan outcome
- `pd_score` → predicted probability of default
- `pd_prediction` → predicted default class

These columns are used to build risk bands and decision rules.

In [8]:
df[['actual_default_flag','pd_score','pd_prediction']].tail(10)

,actual_default_flag,pd_score,pd_prediction
692153,0,0.468976,0
1816200,0,0.233568,0
1270714,0,0.050490,0
1709748,0,0.533128,1
596968,0,0.739552,1
1135476,1,0.222382,0
1601713,1,0.950497,1
344609,0,0.146255,0
874428,1,0.554675,1
1910437,1,0.368561,0


## PD Score Distribution

The distribution of PD scores is reviewed using summary statistics.

This helps understand:

- minimum predicted default risk
- maximum predicted default risk
- average PD score
- spread of borrower risk levels
- quartile ranges

Understanding PD score distribution is important before creating risk bands.

In [9]:
df['pd_score'].describe()

count    40000.000000
mean         0.438078
std          0.215661
min          0.025880
25%          0.263283
50%          0.421281
75%          0.598333
max          0.967616
Name: pd_score, dtype: float64

## Risk Segmentation Using Quantiles

Borrowers are segmented into five risk bands using PD score quantiles.

The five risk bands are:

- Very Low
- Low
- Medium
- High
- Very High

Quantile-based segmentation ensures that each band contains approximately the same number of borrowers.

In [12]:
df['risk_band'] = pd.qcut(df['pd_score'], q=5, labels=['Very Low','Low', 'Medium', 'High', 'Very High'])
df[['pd_score','risk_band']].head(10)

,pd_score,risk_band
824713,0.495415,High
591446,0.504460,High
1789024,0.301881,Low
1084390,0.685799,Very High
1807296,0.739917,Very High
1286568,0.721815,Very High
713788,0.553942,High
2050348,0.275931,Low
1266891,0.943725,Very High
1271617,0.363590,Medium


## Risk Band Distribution

The number of borrowers in each risk band is checked.

Since quantile segmentation is used, each group should contain an approximately equal number of borrowers.

In [13]:
df['risk_band'].value_counts().sort_index()

risk_band
Very Low     8000
Low          8000
Medium       8000
High         8000
Very High    8000
Name: count, dtype: int64

## Risk Band Summary

A risk band summary table is created to evaluate actual default behavior across each PD-based segment.

The summary includes:

- borrower count
- default count
- actual default rate
- minimum PD score
- maximum PD score
- average PD score

This validates whether higher PD bands correspond to higher observed default rates.

In [19]:
risk_band_summary = df.groupby('risk_band').agg(
    borrower_count=('actual_default_flag', 'count'),
    default_count=('actual_default_flag', 'sum'),
    default_rate=('actual_default_flag', 'mean'),
    min_pd_score=('pd_score','min'),
    max_pd_score=('pd_score','max'),
    avg_pd_score=('pd_score','mean')
).reset_index()

risk_band_summary

/var/folders/99/nqyz4s796bx09s6p71mvxd5h0000gn/T/ipykernel_19372/938080201.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  risk_band_summary = df.groupby('risk_band').agg(


,risk_band,borrower_count,default_count,default_rate,min_pd_score,max_pd_score,avg_pd_score
0,Very Low,8000,389,0.048625,0.025880,0.230880,0.155090
1,Low,8000,860,0.107500,0.230932,0.358714,0.294819
2,Medium,8000,1473,0.184125,0.358716,0.486674,0.422031
3,High,8000,2111,0.263875,0.486684,0.641309,0.560484
4,Very High,8000,4027,0.503375,0.641349,0.967616,0.757968


## Create Scorecard-Style Risk Score

A scorecard-style risk score is created from the PD score.

Formula:

`Risk Score = (1 - PD Score) × 1000`

Interpretation:

- Higher PD score → Higher default risk
- Higher risk score → Lower default risk

This makes the model output easier to interpret in a credit risk decisioning context.

In [26]:
df['risk_score'] = ((1-df['pd_score'])*1000).round(0)
df[['pd_score','risk_score','risk_band']].head(10)

,pd_score,risk_score,risk_band
824713,0.495415,505.0,High
591446,0.504460,496.0,High
1789024,0.301881,698.0,Low
1084390,0.685799,314.0,Very High
1807296,0.739917,260.0,Very High
1286568,0.721815,278.0,Very High
713788,0.553942,446.0,High
2050348,0.275931,724.0,Low
1266891,0.943725,56.0,Very High
1271617,0.363590,636.0,Medium


## Risk Score Summary

A risk score summary is created to analyze borrower counts and default rates across score values.

This helps evaluate whether lower scores are associated with higher observed default rates.

In [31]:
score_summary = df.groupby('risk_score').agg(
    borrower_count=('risk_score','count'),
    avg_risk_score=('risk_score','mean'),
    min_risk_score=('risk_score','min'),
    max_risk_score=('risk_score','max'),
    actual_default_rate=('actual_default_flag','mean')
).reset_index()

score_summary

,risk_score,borrower_count,avg_risk_score,min_risk_score,max_risk_score,actual_default_rate
0,32.0,1,32.0,32.0,32.0,1.00
1,36.0,1,36.0,36.0,36.0,1.00
2,38.0,1,38.0,38.0,38.0,1.00
3,39.0,2,39.0,39.0,39.0,1.00
4,40.0,4,40.0,40.0,40.0,0.75
...,...,...,...,...,...,...
931,968.0,1,968.0,968.0,968.0,0.00
932,969.0,6,969.0,969.0,969.0,0.00
933,970.0,1,970.0,970.0,970.0,0.00
934,971.0,3,971.0,971.0,971.0,0.00


## Review Risk Band Categories

The unique risk band categories are checked to confirm that all expected segments were created successfully.

In [32]:
df['risk_band'].unique()

['High', 'Low', 'Very High', 'Medium', 'Very Low']
Categories (5, object): ['Very Low' < 'Low' < 'Medium' < 'High' < 'Very High']

## Preliminary Loan Decision Rule

A simple decision rule is created based on risk band:

- Very Low / Low → Approve
- Medium → Manual Review
- High / Very High → Reject

This demonstrates how PD-based risk segmentation can support lending decisions.

In a real banking environment, final decisions would also consider policy rules, affordability, compliance, and manual underwriting review.

In [33]:
def preliminary_decision(risk_band):
    if risk_band in ['Very Low','Low']:
        return 'Approve'
    elif risk_band == 'Medium':
        return 'Manual Review'
    else:
        return 'Reject'

## Decision Distribution

The number of borrowers assigned to each preliminary decision category is reviewed.

This helps understand portfolio-level approval, review, and rejection distribution.

In [34]:
df['preliminary_decision']=df['risk_band'].apply(preliminary_decision)
df['preliminary_decision'].value_counts()

preliminary_decision
Reject           16000
Approve          16000
Manual Review     8000
Name: count, dtype: int64

## Decision Summary

A decision-level summary is created to evaluate the risk characteristics of each decision group.

The summary includes:

- borrower count
- average PD score
- average risk score
- actual default rate

This helps verify whether the decision rule separates borrowers into meaningful risk groups.

In [35]:
decision_summary = df.groupby('preliminary_decision').agg(
    borrower_count=('actual_default_flag', 'count'),
    avg_pd_score=('pd_score','mean'),
    avg_risk_score=('risk_score', 'mean'),
    actual_default_rate=('actual_default_flag','mean')
).reset_index()

decision_summary

,preliminary_decision,borrower_count,avg_pd_score,avg_risk_score,actual_default_rate
0,Approve,16000,0.224955,775.046812,0.078063
1,Manual Review,8000,0.422031,577.970250,0.184125
2,Reject,16000,0.659226,340.774688,0.383625


## Save Risk Segmentation Outputs

The final risk-segmented dataset and summary tables are saved for downstream analysis.

Saved outputs include:

- full risk segmented borrower output
- risk band summary
- risk score summary
- decision summary

These outputs can be used in later stages for portfolio analysis, Expected Loss estimation, reporting, and business decisioning.

In [37]:
df.to_pickle('../data/risk_segmented_output.pkl')
risk_band_summary.to_csv('../data/risk_band_summary.csv', index=False)
score_summary.to_csv('../data/score_summary.csv', index=False)
decision_summary.to_csv('../data/decision_summary.csv', index=False)

# Conclusion

This notebook converted Probability of Default model outputs into practical credit risk decisioning layers.

Key steps completed:

1. Imported scored PD model output
2. Reviewed PD score distribution
3. Created quantile-based risk bands
4. Validated actual default rates by risk band
5. Created scorecard-style risk scores
6. Built preliminary loan decision rules
7. Summarized default risk by decision category
8. Saved risk segmentation outputs

Key observation:

Borrowers in higher PD score bands showed higher actual default rates, confirming that the PD model provides useful risk separation.

This notebook bridges machine learning predictions with business-level credit risk interpretation and decision support.